# Connecting EKS MCP Server to AgentCore Gateway

This lab deploys a [FastMCP](https://github.com/jlowin/fastmcp) server on Amazon EKS inside a private VPC, then connects it to [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) using managed VPC egress with an internal NLB.

The `routingDomain` parameter tells VPC Lattice to route via the NLB's publicly resolvable DNS, while the target URL uses a domain covered by your ACM public certificate.

For background on VPC egress, routing domains, and certificate requirements, see the [project README](../README.md) and [prerequisites](../00-prerequisites/).

![arch](./images/eks-mcp.png)

## Prerequisites

- Completed [Lab 0](../00-prerequisites/00-vpc-gateway-setup.ipynb) (VPC + AgentCore Gateway deployed)
- Docker running (for CDK container image builds)
- An [ACM public certificate](../00-prerequisites/create-acm-public-certificate.md) for TLS termination on the NLB

## Step 1: Install dependencies and import libraries

In [ ]:
import os
from pathlib import Path

# Change to project root
cwd = Path.cwd()
while cwd != cwd.parent:
    if (cwd / "cdk.json").exists():
        break
    cwd = cwd.parent
os.chdir(cwd)
print(f"Working directory: {os.getcwd()}")

!pip install --force-reinstall -q -r requirements.txt

In [ ]:
import json
import os
import time

import boto3
from utils.utils import get_token

# Restore variables from Lab 0
%store -r ACCOUNT_A_ID
%store -r ACCOUNT_A_PROFILE
%store -r GATEWAY_ID
%store -r GATEWAY_URL
%store -r USER_POOL_ID
%store -r USER_POOL_CLIENT_ID
%store -r TOKEN_ENDPOINT_URL
%store -r OAUTH_SCOPES
%store -r VPC_USW2_ID
%store -r VPC_USW2_PRIVATE_SUBNETS

os.environ["ACCOUNT_A_ID"] = ACCOUNT_A_ID

REGION = "us-west-2"
session = boto3.Session(profile_name=ACCOUNT_A_PROFILE, region_name=REGION)
agentcore = session.client("bedrock-agentcore-control")

# Get Cognito client secret
cognito = session.client("cognito-idp")
client_desc = cognito.describe_user_pool_client(
    UserPoolId=USER_POOL_ID, ClientId=USER_POOL_CLIENT_ID
)
CLIENT_SECRET = client_desc["UserPoolClient"]["ClientSecret"]

print(f"Account:    {ACCOUNT_A_ID}")
print(f"Region:     {REGION}")
print(f"Gateway ID: {GATEWAY_ID}")
print(f"VPC ID:     {VPC_USW2_ID}")

In [ ]:
CERT_ARN = input("ACM public certificate ARN: ")
DOMAIN = input(
    "Domain name covered by the certificate (e.g., api.external.yourcompany.com): "
)

assert CERT_ARN.startswith("arn:aws:acm:"), "Invalid certificate ARN"
assert not DOMAIN.startswith("http"), "Domain should not include http:// or https://"
assert "." in DOMAIN, "Domain must contain at least one dot"

print(f"Cert ARN: {CERT_ARN}")
print(f"Domain:   {DOMAIN}")

## Step 2: Deploy MCP server on EKS

This CDK stack deploys:
- **Shared EKS cluster** with an [NGINX Ingress Controller](https://kubernetes.github.io/ingress-nginx/) behind a single internal NLB (TLS termination on port 443 using your ACM public certificate)
- **Two MCP servers** running FastMCP as Kubernetes Deployments, each with a ClusterIP Service
- **Ingress resource** with path-based routing: `/mcp-server/*` → port 8000, `/stock-mcp/*` → port 8001

A single NLB serves both MCP servers through NGINX path-based routing, instead of one NLB per server.

In [ ]:
# # Deploy shared EKS cluster with NGINX Ingress Controller (skip if already deployed)
# # --exclusively avoids cross-stack update issues
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk deploy SharedEksCluster \
#     -c publicCertArn={CERT_ARN} \
#     --profile {ACCOUNT_A_PROFILE} \
#     --require-approval never \
#     --outputs-file eks-cluster-outputs.json \
#     --exclusively

In [ ]:
# Deploy MCP servers on EKS (ClusterIP services + Ingress resource)
# publicCertArn is needed so CDK synthesizes the McpEks stack (gated by the cert check)
!ACCOUNT_A_ID={ACCOUNT_A_ID} cdk deploy McpEks \
    -c publicCertArn={CERT_ARN} \
    --profile {ACCOUNT_A_PROFILE} \
    --require-approval never \
    --outputs-file eks-mcp-outputs.json

In [ ]:
print("Waiting for NGINX Ingress NLB to be provisioned...")

elbv2_client = session.client("elbv2")
ec2_client = session.client("ec2")

NLB_DNS = None
NLB_SG_ID = None
for attempt in range(20):
    nlbs = elbv2_client.describe_load_balancers()["LoadBalancers"]
    # Look for NLB created by the NGINX Ingress Controller
    # AWS LB Controller names NLBs like "k8s-ingressn-ingressn-<hash>" (truncated from ingress-nginx)
    nginx_nlbs = [
        n
        for n in nlbs
        if n.get("VpcId") == VPC_USW2_ID
        and n["Scheme"] == "internal"
        and n["Type"] == "network"
        and "k8s-ingressn" in n.get("LoadBalancerName", "").lower()
    ]
    if nginx_nlbs:
        nlb = nginx_nlbs[0]
        NLB_DNS = nlb["DNSName"]
        NLB_SG_ID = nlb["SecurityGroups"][0] if nlb.get("SecurityGroups") else None
        break
    print(f"  Waiting... (attempt {attempt + 1}/20)")
    time.sleep(15)

assert (
    NLB_DNS
), "NGINX Ingress NLB not found. Check if the NGINX Ingress Controller is running."

# Ensure the NLB SG allows inbound on port 443 from VPC CIDR
if NLB_SG_ID:
    try:
        ec2_client.authorize_security_group_ingress(
            GroupId=NLB_SG_ID,
            IpPermissions=[
                {
                    "IpProtocol": "tcp",
                    "FromPort": 443,
                    "ToPort": 443,
                    "IpRanges": [
                        {"CidrIp": "10.0.0.0/16", "Description": "Allow TLS from VPC"}
                    ],
                }
            ],
        )
        print(f"Added inbound rule: {NLB_SG_ID} <- TCP 443 from 10.0.0.0/16")
    except ec2_client.exceptions.ClientError as e:
        if "InvalidPermission.Duplicate" in str(e):
            print(f"Inbound rule already exists on {NLB_SG_ID}")
        else:
            raise

print(f"\nNLB DNS (routingDomain): {NLB_DNS}")
print(f"NLB SG:                  {NLB_SG_ID}")

## Step 3: Create AgentCore Gateway target

Create a gateway target using [managed VPC Lattice](../01-managed-lattice/).

- **Target URL** (`https://{DOMAIN}/mcp-server/mcp`) — the NGINX Ingress rewrites `/mcp-server/mcp` to `/mcp` before forwarding to the pod
- **`routingDomain`** (`{NLB_DNS}`) — the NGINX Ingress NLB's publicly resolvable DNS name

Both MCP servers share the same NLB via path-based routing. VPC Lattice routes traffic to the NLB using its DNS while setting the TLS SNI to the target domain, so the TLS handshake succeeds against the NLB's certificate. You do **not** need a public DNS record for the target domain.

> **Security group:** We pass the NLB's security group to `securityGroupIds` so the Resource Gateway ENIs can reach the NLB on port 443.

In [ ]:
TARGET_ENDPOINT = f"https://{DOMAIN}/mcp-server/mcp"

print(f"Target endpoint:  {TARGET_ENDPOINT}")
print(f"Routing domain:   {NLB_DNS}")

managed_lattice_config = {
    "vpcIdentifier": VPC_USW2_ID,
    "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
    "endpointIpAddressType": "IPV4",
    "routingDomain": NLB_DNS,
}
if NLB_SG_ID:
    managed_lattice_config["securityGroupIds"] = [NLB_SG_ID]

response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="eks-mcp-server",
    description="MCP server on EKS via NGINX Ingress and managed VPC egress",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": TARGET_ENDPOINT,
            }
        }
    },
    privateEndpoint={
        "managedLatticeResource": managed_lattice_config,
    },
)

TARGET_ID = response["targetId"]
print(f"\nTarget ID: {TARGET_ID}")
print(f"Status:    {response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(
        gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID
    )
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nTarget is active!")
        print(f"  Managed resources: {target.get('privateEndpoint', {})}")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

## Step 4: Invoke the MCP server through AgentCore Gateway

Get an access token from Cognito, then invoke the MCP server's tools through the gateway.

In [ ]:
token_response = get_token(
    token_endpoint_url=TOKEN_ENDPOINT_URL,
    client_id=USER_POOL_CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string=OAUTH_SCOPES.replace(",", " "),
)
ACCESS_TOKEN = token_response["access_token"]
print(f"Access token obtained (expires in {token_response['expires_in']}s)")

In [ ]:
import requests

headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
}

# List available tools
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
print("Available tools:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# Invoke the "add" tool on the private MCP server
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "eks-mcp-server___add",
            "arguments": {"a": 5, "b": 3},
        },
        "id": 2,
    },
)
print("Result of add(5, 3):")
print(json.dumps(response.json(), indent=2))

## Step 5 (Optional): Connect the Stock Price MCP Server

The CDK stack deployed a second MCP server — a **stock price mock** — routed through the same NGINX Ingress NLB at a different path (`/stock-mcp/mcp`). This demonstrates connecting **multiple MCP servers** through a single load balancer using path-based routing.

In [ ]:
STOCK_TARGET_ENDPOINT = f"https://{DOMAIN}/stock-mcp/mcp"

print(f"Stock target endpoint: {STOCK_TARGET_ENDPOINT}")
print(f"Routing domain:        {NLB_DNS} (same NLB)")

# Same NLB and routingDomain as the first MCP server — different path
stock_lattice_config = {
    "vpcIdentifier": VPC_USW2_ID,
    "subnetIds": VPC_USW2_PRIVATE_SUBNETS,
    "endpointIpAddressType": "IPV4",
    "routingDomain": NLB_DNS,
}
if NLB_SG_ID:
    stock_lattice_config["securityGroupIds"] = [NLB_SG_ID]

stock_response = agentcore.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name="eks-stock-mcp",
    description="Stock price MCP server on EKS via NGINX Ingress (shared NLB)",
    targetConfiguration={
        "mcp": {
            "mcpServer": {
                "endpoint": STOCK_TARGET_ENDPOINT,
            }
        }
    },
    privateEndpoint={
        "managedLatticeResource": stock_lattice_config,
    },
)

STOCK_TARGET_ID = stock_response["targetId"]
print(f"\nStock Target ID: {STOCK_TARGET_ID}")
print(f"Status:          {stock_response['status']}")

In [ ]:
while True:
    target = agentcore.get_gateway_target(
        gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID
    )
    status = target["status"]
    print(f"Status: {status}")
    if status == "READY":
        print("\nStock target is active!")
        break
    if status == "FAILED":
        print(f"\nTarget creation failed: {target.get('statusReasons', [])}")
        break
    time.sleep(30)

In [ ]:
# List stock MCP tools
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
)
tools = response.json().get("result", {}).get("tools", [])
stock_tools = [t for t in tools if t["name"].startswith("eks-stock-mcp___")]
print(f"Stock MCP tools ({len(stock_tools)}):")
for t in stock_tools:
    print(f"  {t['name']}: {t.get('description', '')[:80]}")

# Get a stock price
response = requests.post(
    GATEWAY_URL,
    headers=headers,
    json={
        "jsonrpc": "2.0",
        "method": "tools/call",
        "params": {
            "name": "eks-stock-mcp___get_stock_price",
            "arguments": {"symbol": "AAPL"},
        },
        "id": 2,
    },
)
print("\nAAPL stock price:")
print(json.dumps(response.json(), indent=2))

## Cleanup

1. Delete the gateway target
2. Destroy the CDK stacks

> **Note:** Only destroy the Shared EKS Cluster if you are not running the [REST API notebook](./api-server-gateway-managed.ipynb).

In [ ]:
# # Step 1: Delete gateway targets
# agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
# print(f"Deleting target: {TARGET_ID}")
# while True:
#     try:
#         t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=TARGET_ID)
#         print(f"  Status: {t['status']}")
#         time.sleep(15)
#     except agentcore.exceptions.ResourceNotFoundException:
#         print("  Target deleted.")
#         break

# # Delete stock target (if created in Step 5)
# try:
#     agentcore.delete_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID)
#     print(f"Deleting stock target: {STOCK_TARGET_ID}")
#     while True:
#         try:
#             t = agentcore.get_gateway_target(gatewayIdentifier=GATEWAY_ID, targetId=STOCK_TARGET_ID)
#             print(f"  Status: {t['status']}")
#             time.sleep(15)
#         except agentcore.exceptions.ResourceNotFoundException:
#             print("  Stock target deleted.")
#             break
# except NameError:
#     pass  # Stock target was not created (Step 5 skipped)

In [ ]:
# # Step 2: Destroy CDK stacks
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy McpEks \
#     -c publicCertArn={CERT_ARN} \
#     --profile {ACCOUNT_A_PROFILE} --force

In [ ]:
# # Only destroy EKS cluster if you DO NOT want to run api-server-gateway-managed.ipynb notebook
# !ACCOUNT_A_ID={ACCOUNT_A_ID} cdk destroy SharedEksCluster \
#     -c publicCertArn={CERT_ARN} \
#     --profile {ACCOUNT_A_PROFILE} --force \
#     --output cdk-eks.out